# CHESCA Tool as Round-Robin Agent Coordinator

이 notebook은 기존 commitment mesh를 유지하면서, CHESCA controller 기능을 각 building agent가 사용할 수 있는 planner tool로 재해석합니다. 매 step round-robin으로 한 agent가 임시 팀장이 되고, 그 agent가 CHESCA planner tool로 같은 baseline proposal을 만든 뒤 기존 commitment 협상을 이어갑니다.

## 1. Google Drive 연결

`chesca_vs_mesh` 폴더 전체를 `MyDrive` 바로 아래에 올린 뒤 실행합니다.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/chesca_vs_mesh')
OFFICIAL_DIR = PROJECT_DIR / 'CHESCA-main'
assert (OFFICIAL_DIR / 'checa' / 'agent.py').exists(), f'공식 CHESCA 폴더를 찾을 수 없습니다: {OFFICIAL_DIR}'
assert (PROJECT_DIR / 'src' / 'chesca_vs_mesh' / 'round_robin_commitment_agent.py').exists(), 'round-robin coordinator 코드가 누락되었습니다.'
print('Project:', PROJECT_DIR)
print('Official source:', OFFICIAL_DIR)

Mounted at /content/drive
Project: /content/drive/MyDrive/chesca_vs_mesh
Official source: /content/drive/MyDrive/chesca_vs_mesh/CHESCA-main


## 2. Colab 설치

CityLearn 2.1b12는 프로젝트에 포함된 `third_party` 런타임을 사용합니다. PyPI CityLearn은 설치하지 않습니다.

In [3]:
%pip install -q "gym==0.26.2" "simplejson>=3.19" "xgboost>=1.7,<3"

import sys
VENDORED_CITYLEARN = PROJECT_DIR / 'third_party' / 'CityLearn-2.1b12'
assert (VENDORED_CITYLEARN / 'citylearn' / 'citylearn.py').exists(), f'CityLearn runtime을 찾을 수 없습니다: {VENDORED_CITYLEARN}'
sys.path.insert(0, str(VENDORED_CITYLEARN))

import numpy as np
import pandas as pd
import scipy
import torch
import xgboost
import citylearn

print('numpy:', np.__version__, 'pandas:', pd.__version__, 'scipy:', scipy.__version__)
print('torch:', torch.__version__, 'xgboost:', xgboost.__version__)
print('CityLearn:', citylearn.__version__, citylearn.__file__)
assert citylearn.__version__ == '2.1b12'

numpy: 2.0.2 pandas: 2.2.2 scipy: 1.16.3
torch: 2.11.0+cu128 xgboost: 2.1.4
CityLearn: 2.1b12 /content/drive/MyDrive/chesca_vs_mesh/third_party/CityLearn-2.1b12/citylearn/__init__.py


## 3. 프로젝트 로드와 설정

`chesca_round_robin_commitment_full_mesh`는 action path가 `chesca_commitment_full_mesh`와 같아야 합니다. 차이는 로그에 `coordinator_id`, `coordinator_policy`, `planner_tool`이 남는다는 점입니다.

In [4]:
import sys
sys.path.insert(0, str(PROJECT_DIR / 'src'))

from chesca_vs_mesh import MeshConfig, available_datasets
from chesca_vs_mesh.commitment_mesh_agent import CommitmentConfig
from chesca_vs_mesh.round_robin_evaluation import RoundRobinBenchmarkSuite

print('Available bundled datasets:')
print(available_datasets())

DATASET = 'citylearn_challenge_2023_phase_3_1'
EPISODE_STEPS = None  # None: official schema 전체 기간. 빠른 smoke test는 71 등으로 변경.
TAG = 'round_robin_commitment_v1'
CONTROLLERS = [
    'chesca_official',
    'chesca_commitment_full_mesh',
    'chesca_round_robin_commitment_full_mesh',
]

previous_mesh_config = MeshConfig(
    rounds=3,
    offer_step=0.04,
    target_quantile=0.65,
    peak_weight=1.00,
    ramp_weight=0.32,
    price_weight=0.10,
    carbon_weight=0.08,
)
commitment_config = CommitmentConfig(
    rounds=3,
    offer_step=0.04,
    target_quantile=0.65,
    peak_weight=1.00,
    ramp_weight=0.32,
    price_weight=0.10,
    carbon_weight=0.08,
    debt_weight=0.18,
    recovery_weight=0.42,
    budget_weight=0.12,
    max_debt_soc=0.18,
    soft_budget_soc=0.22,
)
suite = RoundRobinBenchmarkSuite(
    output_directory=PROJECT_DIR / 'results',
    mesh_config=previous_mesh_config,
    commitment_config=commitment_config,
    power_outage_seed=None,
    unavailable_coordinators=None,
)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Available bundled datasets:
['citylearn_challenge_2023_phase_1', 'citylearn_challenge_2023_phase_2_local_evaluation', 'citylearn_challenge_2023_phase_2_online_evaluation_1', 'citylearn_challenge_2023_phase_2_online_evaluation_2', 'citylearn_challenge_2023_phase_2_online_evaluation_3', 'citylearn_challenge_2023_phase_3_1', 'citylearn_challenge_2023_phase_3_2', 'citylearn_challenge_2023_phase_3_3', 'warm_up']


## 4. 단일 schema 동일성 검증

두 commitment controller의 score가 동일해야 정상입니다. 아주 작은 부동소수점 차이는 허용합니다.

In [5]:
result = suite.compare_controllers(
    dataset_name=DATASET,
    controllers=CONTROLLERS,
    episode_steps=EPISODE_STEPS,
    tag=TAG,
)

score_columns = [
    'controller', 'challenge_cost', 'comfort_cost', 'emissions_cost',
    'grid_cost', 'resilience_cost', 'challenge_cost_change_vs_chesca_pct',
    'grid_cost_change_vs_chesca_pct', 'resilience_cost_change_vs_chesca_pct',
    'coordinator_switches', 'unique_coordinators',
]
display(result.summary[[c for c in score_columns if c in result.summary.columns]])
display(result.citylearn_metrics)

base = result.summary.set_index('controller').loc['chesca_commitment_full_mesh']
rr = result.summary.set_index('controller').loc['chesca_round_robin_commitment_full_mesh']
check_columns = ['challenge_cost', 'comfort_cost', 'emissions_cost', 'grid_cost', 'resilience_cost']
deltas = (rr[check_columns] - base[check_columns]).astype(float)
display(deltas.rename('round_robin_minus_commitment_full'))
assert np.allclose(deltas.values, 0.0, atol=1e-9), 'Round-robin coordinator가 commitment full과 다른 성능을 냈습니다.'
print('Round-robin equivalence check passed.')
print('Results saved to:', result.output_directory)

,controller,challenge_cost,comfort_cost,emissions_cost,grid_cost,resilience_cost,challenge_cost_change_vs_chesca_pct,grid_cost_change_vs_chesca_pct,resilience_cost_change_vs_chesca_pct,coordinator_switches,unique_coordinators
0,chesca_official,0.590169,0.130315,0.925583,0.957768,0.570619,0.000000,0.000000,0.000000,NaN,NaN
1,chesca_commitment_full_mesh,0.586544,0.130391,0.926412,0.943842,0.572110,-0.614197,-1.454016,0.261284,NaN,NaN
2,chesca_round_robin_commitment_full_mesh,0.586544,0.130391,0.926412,0.943842,0.572110,-0.614197,-1.454016,0.261284,2206.0,6.0


,controller,carbon_emissions_total,discomfort_proportion,ramping_average,daily_one_minus_load_factor_average,daily_peak_average,annual_peak_average,one_minus_thermal_resilience_proportion,power_outage_normalized_unserved_energy_total,average_score
0,chesca_official,0.925583,0.130315,0.850411,0.958645,0.889024,1.132991,0.788781,0.352457,0.590169
1,chesca_commitment_full_mesh,0.926412,0.130391,0.830527,0.956676,0.887802,1.100362,0.792749,0.351471,0.586544
2,chesca_round_robin_commitment_full_mesh,0.926412,0.130391,0.830527,0.956676,0.887802,1.100362,0.792749,0.351471,0.586544


,round_robin_minus_commitment_full
challenge_cost,0.0
comfort_cost,0.0
emissions_cost,0.0
grid_cost,0.0
resilience_cost,0.0


Round-robin equivalence check passed.
Results saved to: /content/drive/MyDrive/chesca_vs_mesh/results/citylearn_challenge_2023_phase_3_1/round_robin_commitment_v1


## 5. Coordinator 로그 확인

`coordinator_id`가 step마다 round-robin으로 바뀌는지 확인합니다. 이 로그는 나중에 coordinator dropout 실험으로 확장할 때 사용합니다.

In [6]:
rr_negotiations = result.negotiations[
    result.negotiations['controller'] == 'chesca_round_robin_commitment_full_mesh'
].copy()
if rr_negotiations.empty:
    print('Round-robin negotiation log가 없습니다.')
else:
    display(rr_negotiations[['step', 'hour', 'coordinator_id', 'coordinator_policy', 'planner_tool']].head(20))
    display(rr_negotiations.groupby('coordinator_id').size().rename('coordinator_steps'))
    display(rr_negotiations.tail(20))

rr_messages = result.messages[
    result.messages['controller'] == 'chesca_round_robin_commitment_full_mesh'
]
if not rr_messages.empty:
    display(rr_messages[['step', 'round_id', 'coordinator_id', 'sender', 'official_grid', 'proposed_grid', 'debt_soc', 'budget_use_soc']].tail(20))

,step,hour,coordinator_id,coordinator_policy,planner_tool
2163,1,2,1.0,round_robin,chesca_planner_tool
2164,2,3,2.0,round_robin,chesca_planner_tool
2165,3,4,3.0,round_robin,chesca_planner_tool
2166,4,5,4.0,round_robin,chesca_planner_tool
2167,5,6,5.0,round_robin,chesca_planner_tool
2168,6,7,0.0,round_robin,chesca_planner_tool
2169,7,8,1.0,round_robin,chesca_planner_tool
2170,8,9,2.0,round_robin,chesca_planner_tool
2171,9,10,3.0,round_robin,chesca_planner_tool
2172,10,11,4.0,round_robin,chesca_planner_tool


,coordinator_steps
coordinator_id,
0.0,360
1.0,360
2.0,360
3.0,361
4.0,362
5.0,360


,controller,step,hour,active_peers,changed_peers,relief_selected_peers,recovery_selected_peers,debt_created_soc,debt_repaid_soc,extra_discharge_soc,...,mean_budget_use_soc,official_predicted_grid,negotiated_predicted_grid,predicted_grid_delta,district_target,final_shadow_signal,logical_message_count,coordinator_id,coordinator_policy,planner_tool
4306,chesca_round_robin_commitment_full_mesh,2187,4,6,0,0,0,0.00,0.000000e+00,0.00,...,0.233333,7.483839,7.483839,0.000000e+00,9.357734,-0.153196,90,3.0,round_robin,chesca_planner_tool
4307,chesca_round_robin_commitment_full_mesh,2188,5,6,0,0,0,0.00,0.000000e+00,0.00,...,0.233333,5.991721,5.991721,0.000000e+00,9.357734,-0.104240,90,4.0,round_robin,chesca_planner_tool
4308,chesca_round_robin_commitment_full_mesh,2189,6,6,0,0,0,0.00,0.000000e+00,0.00,...,0.220000,6.246651,6.246651,0.000000e+00,9.357734,0.072411,90,5.0,round_robin,chesca_planner_tool
4309,chesca_round_robin_commitment_full_mesh,2190,7,6,2,0,2,0.00,3.469447e-18,0.00,...,0.213333,5.920871,6.052871,1.320000e-01,9.357734,-0.091028,90,0.0,round_robin,chesca_planner_tool
4310,chesca_round_robin_commitment_full_mesh,2191,8,6,0,0,0,0.00,0.000000e+00,0.00,...,0.186667,4.434659,4.434659,0.000000e+00,9.357734,-0.041716,90,1.0,round_robin,chesca_planner_tool
4311,chesca_round_robin_commitment_full_mesh,2192,9,6,0,0,0,0.00,0.000000e+00,0.00,...,0.186667,3.691836,3.691836,4.440892e-16,9.357734,-0.015929,90,2.0,round_robin,chesca_planner_tool
4312,chesca_round_robin_commitment_full_mesh,2193,10,6,3,0,3,0.00,6.938894e-18,0.00,...,0.176667,3.668283,3.806569,1.382865e-01,9.357734,-0.097039,90,3.0,round_robin,chesca_planner_tool
4313,chesca_round_robin_commitment_full_mesh,2194,11,6,0,0,0,0.00,0.000000e+00,0.00,...,0.176667,3.481444,3.481444,0.000000e+00,9.357734,0.042115,90,4.0,round_robin,chesca_planner_tool
4314,chesca_round_robin_commitment_full_mesh,2195,12,6,6,6,0,0.20,0.000000e+00,0.20,...,0.210000,5.269561,4.553561,-7.160000e-01,9.357734,0.089421,90,5.0,round_robin,chesca_planner_tool
4315,chesca_round_robin_commitment_full_mesh,2196,13,6,0,0,0,0.00,0.000000e+00,0.00,...,0.210000,6.179185,6.179185,0.000000e+00,9.357734,0.314090,90,0.0,round_robin,chesca_planner_tool


,step,round_id,coordinator_id,sender,official_grid,proposed_grid,debt_soc,budget_use_soc
77848,2205,2,3.0,4,3.011581,3.171581,0.08,0.18
77849,2205,2,3.0,5,1.598441,1.730441,0.08,0.16
77850,2206,0,4.0,0,1.541802,1.541802,0.06,0.24
77851,2206,0,4.0,1,0.804087,0.804087,0.04,0.22
77852,2206,0,4.0,2,1.131355,1.131355,0.04,0.20
77853,2206,0,4.0,3,2.075204,2.075204,0.02,0.26
77854,2206,0,4.0,4,2.828436,2.828436,0.04,0.22
77855,2206,0,4.0,5,1.781099,1.781099,0.04,0.20
77856,2206,1,4.0,0,1.541802,1.541802,0.06,0.24
77857,2206,1,4.0,1,0.804087,0.804087,0.04,0.22


## 6. Public / Private Cost 동일성 검증

이 셀은 3개 public schema와 3개 private schema를 실행합니다. 시간이 걸리지만, round-robin 구조가 full commitment와 같은 성능을 유지하는지 최종 확인할 수 있습니다.

In [ ]:
RUN_PUBLIC_PRIVATE = True

if RUN_PUBLIC_PRIVATE:
    leaderboard = suite.compare_public_private_costs(
        controllers=CONTROLLERS,
        episode_steps=None,
        tag='paper_public_private_round_robin_commitment_v1',
    )
    display(leaderboard.paper_table)
    display(leaderboard.summary)
    display(leaderboard.runs[[
        'split', 'run_id', 'dataset', 'controller', 'challenge_cost',
        'grid_cost', 'resilience_cost',
    ]])

    pivot = leaderboard.runs.pivot_table(
        index=['split', 'run_id', 'dataset'],
        columns='controller',
        values='challenge_cost',
    )
    pivot['round_robin_minus_commitment_full'] = (
        pivot['chesca_round_robin_commitment_full_mesh']
        - pivot['chesca_commitment_full_mesh']
    )
    display(pivot[['round_robin_minus_commitment_full']])
    assert np.allclose(pivot['round_robin_minus_commitment_full'].values, 0.0, atol=1e-9), 'Public/private에서 round-robin과 commitment full이 달라졌습니다.'
    print('Public/private round-robin equivalence check passed.')
    print('Public/private results saved to:', leaderboard.output_directory)
else:
    print('RUN_PUBLIC_PRIVATE=True로 변경하면 Public/Private Cost 비교를 실행합니다.')